CP4 - IoT

---
Classificação de qualidade de vinho
---
Integrantes do Grupo
```
RM566224 -> Victor Sabelli
RM561671 -> Rafaela Ferreira
RM561996 -> Lucca Gomes
RM561408 -> Gustavo Crevelari
```





In [8]:
# Importação das Bibliotecas
import pandas as pd
import joblib

# Separação de treino e teste
from sklearn.model_selection import train_test_split

# Métricas de avaliação para classificação:
# Matriz de Confusão
from sklearn.metrics import confusion_matrix
# Acurácia
from sklearn.metrics import accuracy_score
# Precisão
from sklearn.metrics import precision_score
# Recall
from sklearn.metrics import recall_score
# F1-Score
from sklearn.metrics import f1_score

# Algoritmos de Classificação
# 1. K-Nearest Neighbors (KNN)
from sklearn.neighbors import KNeighborsClassifier
# 2. Random Forest
from sklearn.ensemble import RandomForestClassifier
# 3. Decision Tree (Árvore de Decisão)
from sklearn.tree import DecisionTreeClassifier
# 4. Logistic Regression (Regressão Logística)
from sklearn.linear_model import LogisticRegression
# 5. SVM (Support Vector Machine)
from sklearn.svm import SVC

In [9]:
# Carregar o dataset de vinhos
dados_wine = pd.read_csv('/content/CLEAN_wine_quality.csv')

# Verificar tamanho e primeiras linhas
print(dados_wine.shape)
print(dados_wine.head())

(647, 13)
    type  fixed acidity  volatile acidity  citric acid  residual sugar  \
0  white            7.2              0.24         0.30             1.6   
1  white            6.8              0.23         0.42             7.4   
2    red            4.6              0.52         0.15             2.1   
3  white            6.7              0.20         0.24             6.5   
4    red            9.9              0.53         0.57             2.4   

   chlorides  free sulfur dioxide  total sulfur dioxide  density    pH  \
0      0.048                 27.0                 131.0  0.99330  3.25   
1      0.044                 56.0                 189.0  0.99580  3.22   
2      0.054                  8.0                  65.0  0.99340  3.90   
3      0.044                 28.0                 100.0  0.99348  3.12   
4      0.093                 30.0                  52.0  0.99710  3.19   

   sulphates  alcohol  quality  
0       0.45     10.5        5  
1       0.48      9.3        6  
2

In [10]:
# Estatísticas descritivas
print(dados_wine.describe())

# Verificar valores únicos da coluna de qualidade
print(dados_wine['quality'].unique())
print(dados_wine['quality'].value_counts())

       fixed acidity  volatile acidity  citric acid  residual sugar  \
count     647.000000        647.000000   647.000000      647.000000   
mean        7.213601          0.345155     0.321453        5.053787   
std         1.325751          0.168139     0.145715        4.395960   
min         4.600000          0.090000     0.000000        0.700000   
25%         6.400000          0.230000     0.240000        1.700000   
50%         7.000000          0.300000     0.310000        2.800000   
75%         7.700000          0.410000     0.400000        7.550000   
max        15.600000          1.330000     1.000000       20.200000   

        chlorides  free sulfur dioxide  total sulfur dioxide     density  \
count  647.000000           647.000000            647.000000  647.000000   
mean     0.056224            30.523184            113.077280    0.994467   
std      0.035660            17.673976             56.135005    0.003013   
min      0.016000             3.000000              7.00

In [11]:
# Converter coluna 'type' em variável numérica (One-Hot Encoding)
dados_wine = pd.get_dummies(dados_wine, columns=['type'], drop_first=True)

# Conferir se a conversão funcionou
print(dados_wine.head())

   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.2              0.24         0.30             1.6      0.048   
1            6.8              0.23         0.42             7.4      0.044   
2            4.6              0.52         0.15             2.1      0.054   
3            6.7              0.20         0.24             6.5      0.044   
4            9.9              0.53         0.57             2.4      0.093   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 27.0                 131.0  0.99330  3.25       0.45   
1                 56.0                 189.0  0.99580  3.22       0.48   
2                  8.0                  65.0  0.99340  3.90       0.56   
3                 28.0                 100.0  0.99348  3.12       0.33   
4                 30.0                  52.0  0.99710  3.19       0.76   

   alcohol  quality  type_white  
0     10.5        5        True  
1      9.3        

In [12]:
# Criar coluna binária para classificação
dados_wine['quality_label'] = dados_wine['quality'].apply(lambda x: 1 if x >= 6 else 0)

# Verificar distribuição
print(dados_wine['quality_label'].value_counts())

quality_label
1    409
0    238
Name: count, dtype: int64


In [13]:
# Selecionar atributos preditores (todas as colunas exceto qualidade e qualidade_label)
X = dados_wine.drop(columns=['quality', 'quality_label'])
y = dados_wine['quality_label']

# Conferir
print(X.head())
print(y.head())

   fixed acidity  volatile acidity  citric acid  residual sugar  chlorides  \
0            7.2              0.24         0.30             1.6      0.048   
1            6.8              0.23         0.42             7.4      0.044   
2            4.6              0.52         0.15             2.1      0.054   
3            6.7              0.20         0.24             6.5      0.044   
4            9.9              0.53         0.57             2.4      0.093   

   free sulfur dioxide  total sulfur dioxide  density    pH  sulphates  \
0                 27.0                 131.0  0.99330  3.25       0.45   
1                 56.0                 189.0  0.99580  3.22       0.48   
2                  8.0                  65.0  0.99340  3.90       0.56   
3                 28.0                 100.0  0.99348  3.12       0.33   
4                 30.0                  52.0  0.99710  3.19       0.76   

   alcohol  type_white  
0     10.5        True  
1      9.3        True  
2     13.1 

In [14]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

In [15]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

In [16]:
svm_model = SVC(kernel='rbf', C=1.0, gamma='scale', random_state=42)
svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)

In [17]:
# Random Forest
print("Acurácia RF:", accuracy_score(y_test, y_pred_rf))
print("Precisão RF:", precision_score(y_test, y_pred_rf))
print("Recall RF:", recall_score(y_test, y_pred_rf))
print("F1 RF:", f1_score(y_test, y_pred_rf))

# SVM
print("Acurácia SVM:", accuracy_score(y_test, y_pred_svm))
print("Precisão SVM:", precision_score(y_test, y_pred_svm))
print("Recall SVM:", recall_score(y_test, y_pred_svm))
print("F1 SVM:", f1_score(y_test, y_pred_svm))

Acurácia RF: 0.7794871794871795
Precisão RF: 0.84
Recall RF: 0.8203125
F1 RF: 0.8300395256916996
Acurácia SVM: 0.6564102564102564
Precisão SVM: 0.6564102564102564
Recall SVM: 1.0
F1 SVM: 0.7925696594427245


In [18]:
# Salvar o modelo Random Forest
joblib.dump(rf_model, 'wine_random_forest_model.pkl')

# Salvar o modelo SVM
joblib.dump(svm_model, 'wine_svm_model.pkl')

['wine_svm_model.pkl']

In [19]:
import pickle

with open("wine_random_forest_model.pkl", "wb") as f:
    pickle.dump(rf_model, f)

with open("wine_svm_model.pkl", "wb") as f:
    pickle.dump(svm_model, f)